In [ ]:
import numpy as np
from transformers import AutoTokenizer
from numpy.linalg import norm
from scipy import spatial
import matplotlib.pyplot as plt
import os

In [ ]:
kv_recomp = np.load("./saved_kv/recompute.npy")
kv_reuse = np.load("./saved_kv/reuse.npy")
kv_selective = np.load("./saved_kv/selective.npy")
input_ids = np.load("./saved_kv/input_ids.npy").tolist()
print(kv_recomp.shape, kv_reuse.shape, kv_selective.shape)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-32B-Instruct", local_files_only=True)
# tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct", local_files_only=True)

In [ ]:
def cos_distance(vec1: np.ndarray, vec2: np.ndarray):
    return spatial.distance.cosine(vec1, vec2)

def l2_distance(vec1: np.ndarray, vec2: np.ndarray):
    return norm(vec1 - vec2)

In [ ]:
def calc_kv_distance(kv1, kv2, layer, korv, seq, head, metric):
    kv1 = kv1[layer, korv, seq, head]
    kv2 = kv2[layer, korv, seq, head]
    dis = np.zeros((kv2.shape[0],))
    for i, (vec1, vec2) in enumerate(zip(kv1, kv2)):
        dis[i] = metric(vec1, vec2)
    return dis

In [ ]:
layer, korv, head, metric = 2, 0, 0, l2_distance
plt.figure(figsize=(12,3))
dis1 = calc_kv_distance(kv_recomp, kv_reuse, layer, korv, 0, head, metric)
plt.plot(dis1, label="Reuse Difference")
dis2 = calc_kv_distance(kv_recomp, kv_selective, layer, korv, 0, head, metric)
plt.plot(dis2, label="EPIC-8 Difference")
plt.legend(loc="upper right")
plt.show()

In [ ]:
def topk_numpy(arr,k,dim):
    idx = np.argpartition(-arr, kth=k, axis=dim)
    idx = idx.take(indices=range(k), axis=dim)
    val = np.take_along_axis(arr, indices=idx, axis=dim)
    sorted_idx = np.argsort(-val, axis=dim)
    idx = np.take_along_axis(idx, indices=sorted_idx, axis=dim)
    val = np.take_along_axis(val, indices=sorted_idx, axis=dim)
    return val, idx.astype(dtype=int)

val, idx = topk_numpy(dis2, 100, 0)
for i in idx[0:40]:
    print(f'{i}:{repr(tokenizer.decode(input_ids[i:i+1]))}')

In [ ]:
def plot_3d_kvcache_distance(cache1, cache2, metric, korv, head=0, interval=6, figname=None):
    ax = plt.figure(figsize=(20,12)).add_subplot(projection='3d')
    ax.set_box_aspect((40, 20, 16))
    ax.view_init(elev=15, azim=-80, roll=0)
    colors = plt.colormaps['viridis_r'](np.linspace(0, 1, cache1.shape[0]))
    seq_len = cache1.shape[4]
    x_indices = list(range(seq_len))
    for layer_i in reversed(range(1, cache1.shape[0], interval)):
        ax.plot(
            x_indices, 
            [layer_i] * seq_len, 
            calc_kv_distance(cache1, cache2, layer_i, korv, 0, head, metric),
            color=colors[layer_i],
        )
    if figname is not None:
        plt.savefig(os.path.join("figures", figname))
    plt.show()

In [ ]:
plot_3d_kvcache_distance(kv_recomp, kv_selective, l2_distance, 0, figname="llama3.1_8b-selective.png")
plot_3d_kvcache_distance(kv_recomp, kv_reuse, l2_distance, 0, figname="llama3.1_8b-reuse.png")

In [ ]:
def calc_norm_ratio(kv1, kv2, korv, head):
    kv1 = kv1[:, korv, 0, head]
    kv2 = kv2[:, korv, 0, head]
    kv1_norm = norm(kv1, axis=-1)
    kv2_norm = norm(kv2, axis=-1)
    ratio = kv1_norm / kv2_norm
    return ratio

In [ ]:
ratio = calc_norm_ratio(kv_recomp, kv_selective, 0, 0)
layer_num, seq_len = ratio.shape
# 计算每个随机变量的均值和标准差
variable_means = np.mean(ratio, axis=1)
variable_stds = np.std(ratio, axis=1)
# 创建图形
plt.figure(figsize=(12, 3))
# 绘制误差棒（用"I"图样标出一个标准差）
plt.errorbar(range(layer_num), variable_means, yerr=variable_stds, 
             fmt='o',  # 不显示数据点，因为我们后面会画折线
             capsize=5,  # 误差棒端帽长度
             capthick=2,  # 误差棒端帽粗细
             elinewidth=2,  # 误差棒线宽
             color='red',  # 误差棒颜色
             alpha=0.7,  # 透明度
             label='1 stddev')
# 绘制平均值点的折线图
plt.plot(range(layer_num), variable_means, 
         marker='o',  # 数据点标记
         markersize=8,  # 标记大小
         linewidth=2,  # 线宽
         color='blue',  # 线条颜色
         label='avg')

plt.xticks(range(layer_num), [f'{i+1}' for i in range(layer_num)])
# 添加网格
plt.grid(True, alpha=0.3, linestyle='--')
# 添加图例
plt.legend(fontsize=10)
# 调整布局
plt.tight_layout()
# 显示图形
plt.savefig(os.path.join("figures", "norm_ratio-qwen3-32b_3.png"))
plt.show()
# np.save("./saved_kv/qwen3-4b_layer-recover.npy", variable_means)

In [ ]:
from mdocdataset import load_mdoc_dataset
import matplotlib.pyplot as plt

amapdata = load_mdoc_dataset("amap")

In [ ]:
# 获取所有sample的documents长度
doc_lengths = [len(sample['documents']) for sample in amapdata]

# 绘制直方图
plt.figure(figsize=(6, 2))
plt.hist(doc_lengths, bins=50, alpha=0.7, color='steelblue')
plt.xlabel('Number of Documents')
plt.ylabel('Frequency')
plt.title('Distribution of Documents Count per Sample')
plt.grid(True, alpha=0.3)
plt.show()

# 打印基本统计信息
print(f"Mean: {np.mean(doc_lengths):.2f}")
print(f"Std: {np.std(doc_lengths):.2f}")
print(f"Min: {min(doc_lengths)}")
print(f"Max: {max(doc_lengths)}")

doc_lengths = []
for sample in amapdata:
    doc_lengths.extend(map(len, sample['documents']))

# 绘制直方图
plt.figure(figsize=(6, 2))
plt.hist(doc_lengths, bins=50, alpha=0.7, color='steelblue')
plt.xlabel('Documents Length')
plt.ylabel('Frequency')
plt.title('Distribution of Document Length')
plt.grid(True, alpha=0.3)
plt.show()

# 打印基本统计信息
print(f"Mean: {np.mean(doc_lengths):.2f}")
print(f"Std: {np.std(doc_lengths):.2f}")
print(f"Min: {min(doc_lengths)}")
print(f"Max: {max(doc_lengths)}")